# Домашнее задание №4

Материалом для практики послужит текст из [книги](https://fb2.top/pisyma-nikolay-erdman-angelina-stepanova-1928-1935-gg-457615), содержащей переписку писателя-драматурга Николая Эрдмана и его возлюбленной Ангелины Степановой, актрисы МХАТа. 
В тексте есть сами письма (с 1928 по 1935 гг.), а также фрагменты из воспоминаний Степановой (опубликованных в 1995 году и относящихся к этому периоду).


Ваша задача — подготовить датасет писем для дальнейшего количественного анализа.

Необходимо сделать pandas датафрейм, в котором буду храниться следующие сведения (столбцы): 
- дата написания письма (желательно в формате YYYY-MM-DD), 
- автор (Степанова или Эрдман), 
- текст письма, 
- лемматизированный текст письма (без знаков препинания), 
- количество слов в письме. 

> Воспоминания сохранять в датафрейм НЕ НУЖНО!
> Сноски вида [23] и переносы строк нужно удалить из текста.

Чтобы извлечь письма и метаданные о них, нужно воспользоваться [регулярными выражениями](https://docs.python.org/3/library/re.html). 

Обратите внимание, что даты оформлены не единообразно. Продумайте все возможные варианты, перепроверьте данные. Адресант письма так же может быть указан не единообразно или не указан вовсе (но, например, есть адресат). Задача — извлечь метаданные для наибольшего количества писем.

После окончания работы над датасетом необходимо вывести df.info(), чтобы можно было посмотреть количество извлечённых дат, а также valuecounts() для столбца с адресантами.

### КРИТЕРИИ

In [70]:
import re
import pandas as pd
import pymorphy3 as pymorphy

MONTHS_RU = {
    'январ': 1, 'феврал': 2, 'март': 3, 'апрел': 4, 'мая': 5, 'июн': 6,
    'июл': 7, 'август': 8, 'сентябр': 9, 'октябр': 10, 'ноябр': 11, 'декабр': 12,
}
MONTHS_ROMAN = {'I': 1,'II': 2,'III': 3,'IV': 4,'V': 5,'VI': 6,'VII': 7,'VIII': 8,'IX': 9,'X': 10,'XI': 11,'XII': 12}

morph = pymorphy.MorphAnalyzer()

def parse_date(s, default_year=None):
    s = s.strip().strip('[]').strip()
    s = re.sub(r'\bг\.|\bгода\b', '', s).strip()

    m = re.search(r'(\d{1,2})\.(\d{1,2})\.(\d{2,4})', s)
    if m:
        d, mo, y = int(m.group(1)), int(m.group(2)), int(m.group(3))
        y = y + 1900 if y < 100 else y
        return f"{y:04d}-{mo:02d}-{d:02d}"

    m = re.search(r'(\d{1,2})/([IVX]+)-(\d{2,4})', s)
    if m:
        d, mo_rom, y = int(m.group(1)), m.group(2), int(m.group(3))
        y = y + 1900 if y < 100 else y
        return f"{y:04d}-{MONTHS_ROMAN[mo_rom]:02d}-{d:02d}"

    m = re.search(r'(\d{1,2})\s+([а-я]+)\s*(\d{2,4})?', s.lower())
    if m and any(m.group(2).startswith(k) for k in MONTHS_RU):
        d = int(m.group(1))
        mo = next(v for k, v in MONTHS_RU.items() if m.group(2).startswith(k))
        y = m.group(3)
        y = (int(y) + 1900 if int(y) < 100 else int(y)) if y else default_year
        if y:
            return f"{y:04d}-{mo:02d}-{d:02d}"

    m = re.search(r'([а-я]+)\s+(\d{2,4})', s.lower())
    if m and any(m.group(1).startswith(k) for k in MONTHS_RU):
        mo = next(v for k, v in MONTHS_RU.items() if m.group(1).startswith(k))
        y = int(m.group(2))
        y = y + 1900 if y < 100 else y
        return f"{y:04d}-{mo:02d}-01"


    m = re.search(r'(\d{1,2})\s+([а-я]+)', s.lower())
    if m and any(m.group(2).startswith(k) for k in MONTHS_RU) and default_year:
        d = int(m.group(1))
        mo = next(v for k, v in MONTHS_RU.items() if m.group(2).startswith(k))
        return f"{default_year:04d}-{mo:02d}-{d:02d}"

    return None

def clean_text(t):
    t = re.sub(r'\[\d+\]', '', t)
    t = re.sub(r'\n+', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t

def lemmatize(text):
    tokens = re.findall(r'[а-я]+', text.lower())
    return ' '.join(morph.parse(t)[0].normal_form for t in tokens)

def detect_author(header, body):
    h = header.lower()
    b = body.lower()
    #Адресант в шапке
    if re.search(r'степанов', h):
        return 'Эрдман'
    if re.search(r'эрдман', h):
        return 'Степанова'
    # Подпись в конце
    tail = b[-30:]
    if re.search(r'николай|н\.р\.|\bн\.\s*$', tail):
        return 'Эрдман'
    if re.search(r'\bлина\b|линуша', tail):
        return 'Степанова'
    return None

with open('erdman.txt', encoding='utf-8') as f:
    raw = f.read()

# Разбиваем по блокам * * *
blocks = re.split(r'\n\s*\*\s*\*\s*\*\s*\n', raw)

# Отслеживаем "текущий год" для писем без года
current_year = None
records = []

for block in blocks:
    block = block.strip()
    if not block:
        continue
    # Отбрасываем Воспомининия по маркеру '[Воспоминания'
    if '[Воспоминания' in block:
        before, _, after = block.partition('[Воспоминания')
        if len(before.strip()) > 50:
            block = before.strip()
        else:
            block = after.strip()
      

    lines = block.split('\n')
    header_lines, body_lines = [], []
    in_header = True
    for line in lines:
        if in_header:
            header_lines.append(line)
            # Шапка заканчивается, когда встречаем строку с датой
            if re.search(r'\d{1,2}[./\-]|\b(19\d{2})\b|\b(январ|феврал|март|апрел|мая|июн|июл|август|сентябр|октябр|ноябр|декабр)', line.lower()):
                in_header = False
        else:
            body_lines.append(line)

    header = '\n'.join(header_lines)
    body = '\n'.join(body_lines).strip()
    if not body:
        header = ""
        body = block

    date = None
    for line in header_lines:
        d = parse_date(line, default_year=current_year)
        if d:
            date = d
            break
    if date:
        year_from_date = int(date[:4])
        current_year = year_from_date

    author = detect_author(header, body)
    body_clean = clean_text(body)
    lemma = lemmatize(body_clean)

    records.append({
        'date': date,
        'author': author,
        'text': body_clean,
        'lemma': lemma,
        'n_words': len(lemma.split()) if lemma else 0,
    })

df = pd.DataFrame(records, columns=['date', 'author', 'text', 'lemma', 'n_words'])
df['date'] = pd.to_datetime(df['date'], errors='coerce')

print(df.info())
print(df['author'].value_counts(dropna=False))

<class 'pandas.DataFrame'>
RangeIndex: 75 entries, 0 to 74
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   date     49 non-null     datetime64[us]
 1   author   62 non-null     str           
 2   text     75 non-null     str           
 3   lemma    75 non-null     str           
 4   n_words  75 non-null     int64         
dtypes: datetime64[us](1), int64(1), str(3)
memory usage: 3.1 KB
None
author
Эрдман       31
Степанова    31
NaN          13
Name: count, dtype: int64


In [72]:
df.to_csv('letters.csv', index=False, encoding='utf-8')

| Что сделано | Баллы |
|---|---:|
| Извлечены тексты писем, собраны в датафрейм с указанными столбцами | 2 |
| Текст очищен и лемматизирован | 1 |
| Автор каждого письма определён: «Степанова» или «Эрдман» (включая случаи, когда автор указан не единообразно или восстанавливается через адресата) | 2 |
| Все имеющиеся даты приведены к единому формату (в оригинале даты не единообразны, например, «1928», «января 1928», «12.02.1928», «12 февраля 1928») | 3 |
| Датасет опубликован на GitHub в отдельном репозитории с файлом README.md (описание источника, структуры столбцов, методов извлечения и очистки, описаны принципы работы с отсутствующими значениями дат). Достаточно прикрепить ссылку на репозиторий. Датасет нужно опубликовать в формате .csv. | 2 |

Для поднятия настроения — басня Эрдмана:

```
СМЕТАНА.

Мы любим подмечать у недругов изъяны,
И направлять на них насмешки остриё.

Однажды молоко спросило у сметаны:
- Скажите, вы еда или питье?
Сметана молвила:
- Оставьте ваши шутки!
Действительно, я где-то в промежутке,
Но ведь важна не эта сторона
Всего важнее то, что я вкусна
И все, как правило, бывают мною сыты.

Вот так же точно и гермафродиты:
Тот, кто на свет их произвел,
Конечно, допустил ужасную небрежность.
Но ведь в конце-концов
существенен не пол,
А классовая принадлежность.
```

Читать другие басни: https://stihi.ru/diary/valery31/2014-10-11